# Episode 14 — Writing a PYNQ driver

In episode 13 we drove the PWM block with raw offsets: `pwm.write(0x10 + 4 * ch, ...)`.
That works, but the register map lives in your head. A **driver** wraps it in a class, so
code reads like this:

```python
ol.pwm_0.frequency = 1000
ol.pwm_0.set_brightness(2, 40)
```

PYNQ picks a driver for each block by its type. A class that subclasses `DefaultIP` and lists
the block's type in `bindto` is used automatically for every matching block in any overlay.

This episode uses the overlay from episode 13 (`pwm_overlay`).

## 1. Find the block's type

In [ ]:
from pynq import Overlay
from pynq.overlays.base import board_variant

chip = {"EDGE-Z10": "z10", "EDGE-Z20": "z20"}[board_variant()]
BIT = "../13_axi_lite_pwm/overlays/{}/pwm_overlay.bit".format(chip)

ol = Overlay(BIT)
print(ol.ip_dict["pwm_0"]["type"])

A Verilog file added as a module reference gets the type `xilinx.com:module_ref:<module>:1.0`.

## 2. The driver

The driver is saved to a file, so any notebook can import it. `%%writefile` writes the
cell to `edge_pwm_driver.py` next to this notebook.

In [ ]:
%%writefile edge_pwm_driver.py
"""PYNQ driver for the edge_pwm block (PYNQ tutorial series, episode 13)."""
from pynq import DefaultIP


class EdgePWM(DefaultIP):
    """4-channel PWM generator.

    pwm.frequency = 1000            # Hz, all channels
    pwm.set_brightness(0, 50)       # channel 0 at 50 %
    pwm.brightness                  # all four, in %
    pwm.off()
    """

    bindto = ["xilinx.com:module_ref:edge_pwm:1.0"]

    ID, ENABLE, PERIOD, DUTY0, COUNT = 0x00, 0x04, 0x08, 0x10, 0x20
    MAGIC = 0x50574D31            # "PWM1"
    CLOCK_HZ = 100_000_000        # the PL clock the block runs on
    CHANNELS = 4

    def __init__(self, description):
        super().__init__(description)
        if self.read(self.ID) != self.MAGIC:
            raise RuntimeError("This block does not answer like an edge_pwm block")
        self._percent = [0.0] * self.CHANNELS

    @property
    def frequency(self):
        """PWM frequency in Hz (the same for all channels)."""
        return self.CLOCK_HZ / self.read(self.PERIOD)

    @frequency.setter
    def frequency(self, hz):
        period = int(round(self.CLOCK_HZ / hz))
        if not 2 <= period < 2 ** 32:
            raise ValueError("frequency out of range")
        self.write(self.PERIOD, period)
        for ch, p in enumerate(self._percent):        # keep the brightness the same
            self.set_brightness(ch, p)

    def set_brightness(self, channel, percent):
        """Channel 0-3, brightness 0-100 %."""
        if not 0 <= channel < self.CHANNELS:
            raise IndexError("channel must be 0-3")
        percent = min(max(float(percent), 0.0), 100.0)
        self._percent[channel] = percent
        self.write(self.DUTY0 + 4 * channel, int(self.read(self.PERIOD) * percent / 100))
        self.write(self.ENABLE, self.read(self.ENABLE) | (1 << channel))

    @property
    def brightness(self):
        return list(self._percent)

    @brightness.setter
    def brightness(self, percent):
        """Set every channel to one value, or give a list of four."""
        values = percent if isinstance(percent, (list, tuple)) else [percent] * self.CHANNELS
        for ch, p in enumerate(values):
            self.set_brightness(ch, p)

    def off(self):
        self.write(self.ENABLE, 0)
        self._percent = [0.0] * self.CHANNELS

## 3. Use it

Import the driver **before** loading the overlay, so PYNQ knows about it when it builds
the overlay's objects. `ol.pwm_0` is now an `EdgePWM`.

In [ ]:
from edge_pwm_driver import EdgePWM

ol = Overlay(BIT)
pwm = ol.pwm_0
print(type(pwm).__name__)

pwm.frequency = 1000
pwm.brightness = [5, 20, 50, 100]
print("frequency", pwm.frequency, "Hz, brightness", pwm.brightness)

## 4. Sliders for the four LEDs

The driver makes the widget code short.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def slider(ch):
    s = widgets.FloatSlider(description="LED %d" % ch, min=0, max=100, step=1,
                            value=pwm.brightness[ch])
    s.observe(lambda change: pwm.set_brightness(ch, change["new"]), names="value")
    return s

display(widgets.VBox([slider(ch) for ch in range(4)]))

## 5. A light chaser with a soft tail

In [ ]:
import time

t0 = time.time()
while time.time() - t0 < 8:
    head = (time.time() - t0) * 4 % 4                 # moves 4 LEDs a second
    pwm.brightness = [100 * max(0.0, 1 - ((head - ch) % 4) / 1.5) ** 2 for ch in range(4)]
    time.sleep(0.02)
pwm.off()

## Try it yourself

Add a `blink(channel, hz)` method that sets the frequency low and the brightness to 50 %,
then test it on channel 3. What happens to the other channels, and why?

In [ ]:
# your code here